### Importing Python Packages and Utility Functions

In [2]:
import sys
print(sys.version)

3.14.3 (main, Feb  3 2026, 15:32:20) [Clang 17.0.0 (clang-1700.6.3.2)]


In [3]:
!pip3 install pandas numpy datetime importlib plotly tqdm

In [4]:
!pip3 install scikit-learn

In [5]:
!pip install shap

In [6]:
import pandas as pd
import numpy as np

import datetime
import os, sys
import importlib

import utils
importlib.reload(utils)


from utils import plot_series, plot_series_with_names, plot_series_bar
from utils import plot_dataframe
from utils import get_universe_adjusted_series, scale_weights_to_one, scale_to_book_long_short
from utils import generate_portfolio, backtest_portfolio
from utils import match_implementations

import plotly.graph_objects as go

### Data Loading and Structure

The dataset consists of three key files:


- **features.parquet**: A DataFrame containing **22 stock features** for each trading day from **2005 to 2025**, stored in a **columnar fashion**. For example, `features["macd"]` will return the `macd` of shape **(5068, 2167)**. All **22** features are guaranteed to have the same shape.

- **universe.parquet**: A DataFrame of the same shape as `features["macd"]`, containing `0` and `1` values, where `1` indicates that the stock is tradable on that day.

- **returns.parquet**: A DataFrame of shape **(3775, 2167)** containing **daily stock returns** from **2005 to 2019**. You will **never receive** the returns for the testing period i.e. from **2020 to 2025**.

### Data Organization:
- **Columns** represent stock identifiers, ranging from **1 to 2167** in increasing order.
  
- **Rows** represent trading days when the market was open.  


In [7]:
# This directory can be used if you're working on a Kaggle Notebook inside the competition
# Change the directory as per your requirements if you're working somewhere else
data_dir = "./"

features = pd.read_parquet(os.path.join(data_dir, "features.parquet"))

universe = pd.read_parquet(os.path.join(data_dir, "universe.parquet"))
 
returns = pd.read_parquet(os.path.join(data_dir, "returns.parquet"))

### Benchmark Strategy: Vectorized Portfolio Generation

In this step we will write a vectorized code to generate our strategy portfolio for all trading days at once without a `for` loop. For doing this we will use direct operations on feature dataframes. But, here's one caution. It is very easy to look into the future feature data when you're using dataframes to construct your portfolio at once.

Below is a vectorized implementation of the benchmark strategy [which you see on Kaggle]. Make sure to note the `alpha.shift(1)` operation in the code! This is to make sure you use only feature values upto the last trading day to construct today's portfolio weights.

You can change the code below to implement your own strategy. Since this vectorized implementation is faster, you can try various versions of your strategy and see their performances.

#### Inputs:
- `entire_features :pd.DataFrame`: Historical feature data (MultiIndex columns: Feature Names, Stock Identifiers).
- `universe: pd.DataFrame`: Binary DataFrame indicating tradable stocks per day.
- `start_date`, `end_date` :`str`: Backtest period in `'YYYY-MM-DD'` format.

#### Output:
- `portfolio(pd.DataFrame)`: Normalized portfolio weights for each stock per day in the specified date range.


In [8]:
# # A Benchmark Strategy for your reference: 
# # This is the code used to generate the Benchmark submission you see in the Kaggle Leaderboard

# # This strategy shows how you can combine different features 
# import a23_xgb
# from a23_xgb import train_and_predict

# def generate_portfolio_vectorized(
#     entire_features: pd.DataFrame,
#     returns: pd.DataFrame,
#     universe: pd.DataFrame,
#     start_date: str,
#     end_date: str,
#     train_end: str = "2018-12-31",
#     test_start: str = "2019-01-01",
#     test_end: str = "2019-12-31",
#     n_long: int = 10,          # ← back to 10 (3 was too concentrated)
#     n_short: int = 10,
#     smooth: float = 0.5,       # EMA blend: new_w = smooth*signal + (1-smooth)*prev_w
#                                # 0.7 → ~30% of yesterday's portfolio carried forward
#                                # Lower = smoother / less turnover (try 0.5–0.8)
# ):

#     # Validate date format
#     try:
#         start_dt = datetime.datetime.strptime(start_date, '%Y-%m-%d')
#         end_dt = datetime.datetime.strptime(end_date, '%Y-%m-%d')
#         cutoff_date = datetime.datetime.strptime('2005-01-01', '%Y-%m-%d')
#     except ValueError:
#         raise ValueError("start_date and end_date must be strings in 'YYYY-MM-DD' format.")

#     # Ensure start_date is before end_date
#     if start_dt >= end_dt:
#         raise ValueError("start_date must be earlier than end_date.")

#     # Ensure start_date is not before '2005-01-01'
#     if start_dt < cutoff_date:
#         raise ValueError("start_date must be later than '2005-01-01'.")

#     # Get trading days within the date range
#     trading_days = universe.index[(universe.index >= start_dt) & (universe.index <= end_dt)]

#     if len(trading_days) == 0:
#         raise ValueError("No Trading Days in the specified dates")
    
#     portfolio = 0

#     universe_boolean = universe.loc[:end_date].astype(bool)
    
#     features_ = entire_features.loc[:end_date]

#     #vector of weights based on OBV with dates (rows are dates, columns are different stocks)
#     #will eventually use ML to learn f(Xt) = yt+1
#     #ML will return TxN matrix 

   
#   # ── 1. Train GP + Model → get scores ────────────────────────────
#     ml_scores = train_and_predict(
#         entire_features,
#         returns,
#         universe,
#         train_end  = train_end,
#         test_start = test_start,
#         test_end   = test_end,
#     )

#     # ── 2. Build raw long-top-N / short-bottom-N weights per day ─────────────
#     book    = n_long + n_short
#     w_long  =  1.0 / book   # +0.05 with 10/10
#     w_short = -1.0 / book   # -0.05 with 10/10

#     raw = pd.DataFrame(0.0, index=ml_scores.index, columns=universe.columns)

#     for date in ml_scores.index:
#         row = ml_scores.loc[date].dropna()
#         if len(row) < book:
#             continue
#         s = row.sort_values()
#         raw.loc[date, s.index[:n_short]]  = w_short   # bottom n_short → short
#         raw.loc[date, s.index[-n_long:]]  = w_long    # top    n_long  → long

#     # ── 3. EMA weight smoothing to reduce turnover ────────────────────────────
#     # new_weight[t] = smooth * raw[t] + (1 - smooth) * new_weight[t-1]
#     # Then rescale so constraints (sum_abs=1, dollar-neutral) are preserved.
#     smoothed = pd.DataFrame(0.0, index=raw.index, columns=raw.columns)
#     prev = pd.Series(0.0, index=raw.columns)

#     for date in raw.index:
#         blended = smooth * raw.loc[date] + (1 - smooth) * prev

#         # Rescale: split long/short sides and normalise each to ±0.5 book
#         pos = blended.clip(lower=0)
#         neg = blended.clip(upper=0)
#         pos_sum = pos.sum()
#         neg_sum = neg.abs().sum()

#         if pos_sum > 1e-9:
#             pos = pos / pos_sum * 0.5   # long book sums to 0.5
#         if neg_sum > 1e-9:
#             neg = neg / neg_sum * 0.5   # short book sums to 0.5 (abs)
#             neg = -neg

#         blended = pos + neg
#         smoothed.loc[date] = blended
#         prev = blended

#     # ── 4. Force strictly 0 outside universe and clip to requested dates ─────
#     smoothed = smoothed.fillna(0.0)
    
#     # Safe boolean mask matching the universe DataFrame strictly
#     uv_mask = universe.fillna(0).astype(bool)
#     smoothed = smoothed.where(uv_mask, 0.0)
    
#     return smoothed.loc[start_date:end_date]


In [9]:
    # signal1 = features["on_balance_volume"].shift(1)
    # signal1 = signal1.where(universe_boolean, np.nan) #remove stocks that aren't tradable for the day
    # signal1 = signal1.rank(axis=1, method="min", ascending=True) #instead of dealing with raw values, rank them 
    # signal1 = signal1.sub(signal1.mean(axis=1), axis=0) #de-means it, subtracts mean from each value
    # signal1 = signal1.div(signal1.abs().sum(axis=1), axis=0) #scale weights to total 1 for each date
    # signal1 = signal1.fillna(0)

    # signal2 = features["ultimate_oscillator"].shift(1)
    # signal2 = signal2.where(universe_boolean, np.nan)
    # signal2 = signal2.rank(axis=1, method="min", ascending=True)
    # signal2 = signal2.sub(signal2.mean(axis=1), axis=0)
    # signal2 = signal2.div(signal2.abs().sum(axis=1), axis=0)
    # signal2 = signal2.fillna(0)

    # portfolio = signal1 - signal2

    # portfolio = portfolio.div(portfolio.abs().sum(axis=1), axis=0) #scale weights to 1 again
    
    # return portfolio.fillna(0).loc[start_date:end_date]

2 step process from now - 

In [10]:
import datetime
import numpy as np
import pandas as pd
import importlib

# Choose your model here:
import a23_xgb
importlib.reload(a23_xgb)
from a23_xgb import train_and_predict

# import a23_xgb
# importlib.reload(a23_xgb)
# from a23_xgb import train_and_predict

# ─────────────────────────────────────────────────────────────────────────────
# TWO-STEP PROCESS
# ─────────────────────────────────────────────────────────────────────────────

def run_ml_pipeline(
    entire_features: pd.DataFrame,
    returns: pd.DataFrame,
    universe: pd.DataFrame,
    train_end: str = "2017-12-31",
    test_start: str = "2005-01-04",
    test_end: str = "2019-12-31"
) -> pd.DataFrame:
    """
    Step 1: Run the heavy machine learning pipeline 
    Returns a dataframe of ML scores (predicted returns or ranks) for every stock.
    Run this cell ONCE.
    """
    print("Running Machine Learning Pipeline... This might take a few minutes.")
    ml_scores = train_and_predict(
        entire_features,
        returns,
        universe,
        train_end  = train_end,
        test_start = test_start,
        test_end   = test_end,
    )
    return ml_scores

In [11]:
# def build_portfolio_from_scores(
#     ml_scores: pd.DataFrame,
#     universe: pd.DataFrame,
#     start_date: str,
#     end_date: str,
#     n_long: int = 10,
#     n_short: int = 10,
#     smooth: float = 0.5,
# ) -> pd.DataFrame:
#     """
#     Dollar-neutral, unit-capital portfolio from pre-computed scores.

#     Fixes:
#     1) Apply universe mask before final portfolio sizing.
#     2) Use equal-weight long/short buckets so max |w| <= 0.1 when
#        n_long >= 5 and n_short >= 5.
#     3) If there are not enough tradable names on either side for a day,
#        return a zero row instead of violating constraints.
#     """

#     # ── Date validation ───────────────────────────────────────────────────────
#     try:
#         start_dt  = datetime.datetime.strptime(start_date, "%Y-%m-%d")
#         end_dt    = datetime.datetime.strptime(end_date,   "%Y-%m-%d")
#         cutoff_dt = datetime.datetime.strptime("2005-01-01", "%Y-%m-%d")
#     except ValueError:
#         raise ValueError("Dates must be strings in 'YYYY-MM-DD' format.")

#     if start_dt >= end_dt:
#         raise ValueError("start_date must be before end_date.")
#     if start_dt < cutoff_dt:
#         raise ValueError("start_date must be >= 2005-01-01.")

#     # Align universe to the score matrix
#     universe = universe.reindex(index=ml_scores.index, columns=ml_scores.columns)
#     uv_mask = universe.fillna(False).astype(bool)

#     trading_days = ml_scores.index[
#         (ml_scores.index >= start_dt) & (ml_scores.index <= end_dt)
#     ]
#     if len(trading_days) == 0:
#         raise ValueError("No trading days in the specified date range.")

#     # ── Raw signal: top/bottom names per day ───────────────────────────────────
#     raw = pd.DataFrame(0.0, index=ml_scores.index, columns=ml_scores.columns)

#     for date in ml_scores.index:
#         row = ml_scores.loc[date].dropna()

#         # only keep tradable names on that date
#         tradable = row.index[uv_mask.loc[date, row.index]]
#         row = row.loc[tradable]

#         if len(row) < (n_long + n_short):
#             continue

#         s = row.sort_values()
#         raw.loc[date, s.index[:n_short]] = -1.0
#         raw.loc[date, s.index[-n_long:]] = 1.0

#     # ── EMA smoothing on the raw signal ───────────────────────────────────────
#     smoothed = pd.DataFrame(0.0, index=raw.index, columns=raw.columns)
#     prev = pd.Series(0.0, index=raw.columns)

#     for date in raw.index:
#         # smooth the signal first
#         blended = smooth * raw.loc[date] + (1 - smooth) * prev

#         # mask BEFORE constructing the final portfolio
#         blended = blended.where(uv_mask.loc[date], 0.0)

#         # choose long and short candidates from the blended signal
#         long_candidates = blended[blended > 0].sort_values(ascending=False)
#         short_candidates = blended[blended < 0].sort_values(ascending=True)

#         # Need at least 5 names per side to satisfy max |w| <= 0.1
#         long_n = min(n_long, len(long_candidates))
#         short_n = min(n_short, len(short_candidates))

#         if long_n < 5 or short_n < 5:
#             # zero row is allowed; it will not fail the non-zero constraint
#             smoothed.loc[date] = 0.0
#             prev = pd.Series(0.0, index=raw.columns)
#             continue

#         long_names = long_candidates.index[:long_n]
#         short_names = short_candidates.index[:short_n]

#         day = pd.Series(0.0, index=raw.columns)
#         day.loc[long_names] = 0.5 / long_n
#         day.loc[short_names] = -0.5 / short_n

#         # final safety mask
#         day = day.where(uv_mask.loc[date], 0.0)

#         smoothed.loc[date] = day
#         prev = day

#     smoothed = smoothed.fillna(0.0)
#     return smoothed.loc[start_date:end_date]

trying new rank hysteresis - Here is exactly how the math plays out if n_long=10 and buffer=5:

The script first checks the Top 15 stocks (n_long + buffer).
Let's say 7 of those Top 15 stocks were already in your portfolio yesterday. It says: "Great, I will keep those 7 stocks to save on trading fees."
Now the script says: "I have 7 stocks, but my target n_long is 10. I need to find 3 new stocks." (slots_to_fill = n_long - len(kept_longs))
It goes back to the absolute top of the rankings (#1, #2, #3...) and picks the 3 highest-ranked stocks that you don't already own.
In the end, you have exactly 10 final stocks.

In [12]:
def generate_portfolio_vectorized(
    ml_scores: pd.DataFrame,
    universe: pd.DataFrame,
    start_date: str,
    end_date: str,
    n_long: int = 10,
    n_short: int = 10,
    buffer: int = 5,
    rebalance_every: int = 1,
) -> pd.DataFrame:
    """
    Dollar-neutral, unit-capital portfolio using RANK HYSTERESIS (Sticky Weights).
    
    Instead of swapping a stock out the moment it drops from rank #10 to #11, 
    the model holds onto it until its rank drops below (n_long + buffer) e.g., #15.
    If an open slot is created, it takes the highest-ranked available stock.
    This drastically reduces transaction fees/turnover without needing EMA smoothing.
    """
    import datetime
    try:
        start_dt  = datetime.datetime.strptime(start_date, "%Y-%m-%d")
        end_dt    = datetime.datetime.strptime(end_date,   "%Y-%m-%d")
    except ValueError:
        raise ValueError("Dates must be strings in 'YYYY-MM-DD' format.")

    universe = universe.reindex(index=ml_scores.index, columns=ml_scores.columns)
    uv_mask = universe.fillna(False).astype(bool)

    portfolio = pd.DataFrame(0.0, index=ml_scores.index, columns=ml_scores.columns)
    
    prev_longs = set()
    prev_shorts = set()
    days_since_rebalance = 0
    
    for date in ml_scores.index:
        row = ml_scores.loc[date]
        active_row = row[uv_mask.loc[date]].dropna()
        
        if len(active_row) < (n_long + buffer + n_short + buffer):
            prev_longs = set()
            prev_shorts = set()
            days_since_rebalance = 0
            continue
            
        # ─── REBALANCE CHECK ───
        force_trade = False
        if not prev_longs or not prev_shorts:
            force_trade = True
        else:
            # If any held stock exits the universe today, we MUST rebalance to avoid a ValueError
            if not prev_longs.issubset(active_row.index) or not prev_shorts.issubset(active_row.index):
                force_trade = True
                
        if not force_trade and (days_since_rebalance % rebalance_every != 0):
            # Skip trading entirely: just hold yesterday's exact positions
            current_longs = prev_longs
            current_shorts = prev_shorts
            days_since_rebalance += 1
        else:
            # Time to rebalance (or forced to)!
            # Sort descending: best long targets are at the TOP, best short targets are at the BOTTOM
            s = active_row.sort_values(ascending=False)
            
            # ─── LONG HYSTERESIS ───
            # Identify the acceptable top zone
            long_candidates = set(s.index[:(n_long + buffer)])
            # Keep stocks that are BOTH currently in portfolio AND in the top buffer zone
            kept_longs = prev_longs.intersection(long_candidates)
            
            slots_to_fill = n_long - len(kept_longs)
            new_longs = []
            for stock in s.index:
                if slots_to_fill <= 0:
                    break
                if stock not in kept_longs:
                    new_longs.append(stock)
                    slots_to_fill -= 1
                    
            current_longs = kept_longs.union(new_longs)
            
            # ─── SHORT HYSTERESIS ───
            # Identify the acceptable bottom zone
            short_candidates = set(s.index[-(n_short + buffer):])
            # Keep stocks that are BOTH currently in portfolio AND in the bottom buffer zone
            kept_shorts = prev_shorts.intersection(short_candidates)
            
            slots_to_fill = n_short - len(kept_shorts)
            new_shorts = []
            # Search backward from the worst scores
            for stock in s.index[::-1]:
                if slots_to_fill <= 0:
                    break
                if stock not in kept_shorts:
                    new_shorts.append(stock)
                    slots_to_fill -= 1
                    
            current_shorts = kept_shorts.union(new_shorts)
            days_since_rebalance = 1

        # ─── ALLOCATE WEIGHTS ───
        w_long = 0.5 / n_long if n_long > 0 else 0
        w_short = -0.5 / n_short if n_short > 0 else 0
        
        portfolio.loc[date, list(current_longs)] = w_long
        portfolio.loc[date, list(current_shorts)] = w_short
        
        prev_longs = current_longs
        prev_shorts = current_shorts
        
    # Return cleanly clipped date range
    return portfolio.fillna(0.0).loc[start_date:end_date]



In [13]:
import importlib
import feature_engineering_ogog
importlib.reload(feature_engineering_ogog)

# Pass your massive original features DataFrame into the engineer function
cleaned_features = feature_engineering_ogog.engineer_features(features)

Feature engineering complete. Returning 22 total features.


### Generate your portfolio using the `generate_portfolio_vectorized` function you wrote above

In [14]:
ml_scores = run_ml_pipeline(
    cleaned_features, returns, universe,
    train_end  = "2019-12-31",
    test_start = "2005-01-04",
    test_end   = "2025-12-31"
)

Running Machine Learning Pipeline... This might take a few minutes.
[1/5] Extracting training features …
[2/5] Flattening …
      Training obs: 3,711,857  |  features: 22
[3/5] Genetic Programming (5 generations) …
    |   Population Average    |             Best Individual              |
---- ------------------------- ------------------------------------------ ----------
 Gen   Length          Fitness   Length          Fitness      OOB Fitness  Time Left
   0    11.08       0.00517666        7        0.0173296       0.00927074      2.20m
   1     2.24        0.0112133        2        0.0157219       0.00859294     50.95s
   2     1.46        0.0136999        2        0.0157952       0.00823457     39.27s
   3     1.00         0.013759        1        0.0150226       0.00431115     18.10s
   4     1.02        0.0138366        1        0.0155288      0.000153734      0.00s
      GP done.  Best programs: 6
[4/5] Training XGBoost regressor …
      XGBoost training complete.
      Calculat

In [40]:
# full_portfolio = build_portfolio_from_scores(
#     ml_scores, universe,
#     start_date = "2005-01-04",
#     end_date   = "2019-12-31",
#     n_long     = 40,
#     n_short    = 40,
#     smooth     = 0.4,
# )
aligned_scores = ml_scores.shift(1)
full_portfolio = generate_portfolio_vectorized(
    aligned_scores, universe,
    start_date = "2005-01-04",
    end_date   = "2025-12-31",
    n_long = 40,
    n_short = 40,
    buffer = 27,
    rebalance_every = 1
)
full_portfolio = full_portfolio.reindex(index=universe.index, columns=universe.columns).fillna(0.0)


In [41]:
benchmark_portfolio_vectorized = full_portfolio


### Backtest your portfolio generated using vectorized code

Note that you can backtest your portfolio till `2019-12-31` since this is the last date in the training period. You don't have access to returns after this date. 

In [42]:
# ── In-sample PnL (2005–2017) ────────────────────────────────────────────────
print("IN-SAMPLE (2005–2017)")

sr_vectorized, pnl_vectorized = backtest_portfolio(
    full_portfolio.loc["2005-01-04":"2017-12-31"],
    returns.loc["2005-01-04":"2017-12-31"],
    universe.loc["2005-01-04":"2017-12-31"],
    plot_=True, print_=True,
)

IN-SAMPLE (2005–2017)
Gross Sharpe Ratio:  5.66
Net Sharpe Ratio:  5.409
Turnover %:  89.408


In [43]:
# ── Out-of-sample PnL (2018–2019) ────────────────────────────────────────────
print("OUT-OF-SAMPLE (2018–2019)")

sr_vectorized, pnl_vectorized = backtest_portfolio(
    full_portfolio.loc["2018-01-01":"2019-12-31"],
    returns.loc["2018-01-01":"2019-12-31"],
    universe.loc["2018-01-01":"2019-12-31"],
    plot_=True, print_=True,
)

OUT-OF-SAMPLE (2018–2019)
Gross Sharpe Ratio:  3.586
Net Sharpe Ratio:  3.28
Turnover %:  95.482


### Benchmark Strategy: Iterative Portfolio Generation

Although the Vectorized Function generated the portfolio very quickly, it is very easy to look into the future data if you are not careful. For instance, remove the `shift(1)` operation and see the performance of the portfolio 😊. Hence, if your vectorized code has a forward bias [lookahead bias], you may see very high [or very low] sharpe ratios which may never be realised in real trading.

To avoid making these mistakes, we simulate our portfolio in a daily iterative fashion, where we call the `get_weights(features: pd.DataFrame, today_universe: pd.Series) -> dict[str, float]` function with **only** the past features data and the current day's trading universe. 

### Function Inputs:
- `features(pd.DataFrame)`:  
  - Contains various stock features indexed by date and stock identifiers.  
  - The features are structured in a **MultiIndex format**, where level 0 represents **feature names** (e.g., "macd", "volatility_60"), and level 1 represents **stock identifiers** (e.g., "1", "2", ..., "2167").  

- `today_universe(pd.Series)`:  
  - A series indicating which stocks can be traded on the current day.  
  - Contains **binary values (0 or 1)**, where **1** means a stock is **tradable**, and **0** means it is not.

You have to change this code, and write your own strategy code inside this function. Make sure it follows the same semantics as explained above.


In [44]:
# A Benchmark Strategy for your reference: 
# This is the code used to generate the Benchmark submission you see in the Kaggle Leaderboard

# This strategy shows how you can combine different features 
def get_weights(features: pd.DataFrame, today_universe: pd.Series) -> dict[str, float]:
    """
    Stateful iterative portfolio constructor.
    Pulls today's ML predictions from global 'ml_scores' and strictly enforces 
    Rank Hysteresis (Sticky Weights) across independent daily calls.
    """
    # 1. Initialize persistent state bounds on the very first day
    if not hasattr(get_weights, "prev_longs"):
        get_weights.prev_longs = set()
        get_weights.prev_shorts = set()
        get_weights.days_since_rebalance = 0

    # Hyperparameters (Matched precisely to your vectorized settings in Cell 19!)
    n_long = 40
    n_short = 40
    buffer = 27
    rebalance_every = 1

    # 2. Safety catches for empty un-predictable days
    if features.shape[0] == 0:
        return (today_universe * 1).replace(0, np.nan).dropna().fillna(0).to_dict()
    
    today_date = features.index[-1]
    
    # If the date is earlier than your ML predictions (e.g. burn-in data), yield empty
    if 'ml_scores' not in globals() or today_date not in ml_scores.index:
         return {}
         
    # 3. Pull explicitly predicted scores for *today only*
    row = ml_scores.loc[today_date]
    uv_mask = today_universe.fillna(0).astype(bool)
    active_row = row[uv_mask].dropna()
    
    if len(active_row) < (n_long + buffer + n_short + buffer):
        get_weights.prev_longs = set()
        get_weights.prev_shorts = set()
        get_weights.days_since_rebalance = 0
        return {}

    # 4. Check whether we are mathematically forced to rebalance today
    force_trade = False
    if not get_weights.prev_longs or not get_weights.prev_shorts:
        force_trade = True
    else:
        # If any held stock exits the universe today, we MUST trade to avoid backtest crashes
        if not get_weights.prev_longs.issubset(active_row.index) or not get_weights.prev_shorts.issubset(active_row.index):
            force_trade = True
            
    if not force_trade and (get_weights.days_since_rebalance % rebalance_every != 0):
        # SKIP DAY: Freeze all holdings and perfectly roll over yesterday's positions
        current_longs = get_weights.prev_longs
        current_shorts = get_weights.prev_shorts
        get_weights.days_since_rebalance += 1
    else:
        # REBALANCE DAY: Deploy full Hysteresis buffer zones!
        s = active_row.sort_values(ascending=False)
        
        # --- LONG HYSTERESIS ---
        long_candidates = set(s.index[:(n_long + buffer)])
        kept_longs = get_weights.prev_longs.intersection(long_candidates)
        slots_to_fill = n_long - len(kept_longs)
        
        new_longs = []
        for stock in s.index:
            if slots_to_fill <= 0: break
            if stock not in kept_longs:
                new_longs.append(stock)
                slots_to_fill -= 1
        current_longs = kept_longs.union(new_longs)
        
        # --- SHORT HYSTERESIS ---
        short_candidates = set(s.index[-(n_short + buffer):])
        kept_shorts = get_weights.prev_shorts.intersection(short_candidates)
        slots_to_fill = n_short - len(kept_shorts)
        
        new_shorts = []
        for stock in s.index[::-1]:
            if slots_to_fill <= 0: break
            if stock not in kept_shorts:
                new_shorts.append(stock)
                slots_to_fill -= 1
        current_shorts = kept_shorts.union(new_shorts)
        
        get_weights.days_since_rebalance = 1

    # 5. Overwrite the state for tomorrow's call
    get_weights.prev_longs = current_longs
    get_weights.prev_shorts = current_shorts

    # 6. Allocate exact Unit-Capital Dictionary Weights
    w_long = 0.5 / n_long if n_long > 0 else 0
    w_short = -0.5 / n_short if n_short > 0 else 0
    
    weights = {}
    for cl in current_longs: weights[cl] = w_long
    for cs in current_shorts: weights[cs] = w_short

    return weights


### Generate your portfolio using the `generate_portfolio` and `get_weights` function you wrote above

Since this function is iteratively called on every trading day, it takes a lot of time [about 40 mintues] to generate the entire portfolio dataframe from `2005-01-03` to `2025-02-07`. Hence, to show an example we call it only for a one year period from `2010-01-01` to `2010-12-31`.

In [45]:
#generate_portfolio in utils.py, has for loop that goes through each day and gets_weights everyday to return portfolio
#portfolio is TxN matrix with values as weights
benchmark_portfolio = generate_portfolio(
    get_weights,
    features,
    universe,
    "2010-01-01",
    "2010-12-31",
)

100%|██████████| 252/252 [01:34<00:00,  2.66it/s]


### Backtest your portfolio

Note that you can backtest your portfolio till `2019-12-31` since this is the last date in the training period. You don't have access to returns after this date. 

In [46]:
sr, pnl = backtest_portfolio(benchmark_portfolio.loc["2010"], returns.loc["2010"], universe.loc["2010"], True, True)

Gross Sharpe Ratio:  7.849
Net Sharpe Ratio:  7.473
Turnover %:  91.389


You can also check the performance of your vectorized portfolio in this period to see if they match!

In [47]:
sr, pnl = backtest_portfolio(benchmark_portfolio_vectorized.loc["2010"], returns.loc["2010"], universe.loc["2010"], True, True)

Gross Sharpe Ratio:  7.875
Net Sharpe Ratio:  7.498
Turnover %:  91.349


### Comparing Iterative and Vectorized Portfolio Implementations
This function evaluates **iterative** and **vectorized** portfolio generation methods by comparing their PnL correlation. If **correlation ≥ 0.98**, both implementations are considered equivalent.
#### Steps:
1. Selects a random start date.
2. Generates portfolios using both methods.
3. Backtests portfolios and computes PnLs.
4. Validates PnL correlation.
#### Criteria:
- **Pass:** Correlation **≥ 0.98** (implementations match).
- **Fail:** Correlation **< 0.98** (error raised).
#### Inputs:
- `contestant_get_weights`: Function for portfolio weights.
- `contestant_vectorized_portfolio`: A Pandas DataFrame containing portfolio weights generated using Vectorized Implementation
- `entire_features`: Feature data (MultiIndex columns).
- `universe`: Tradable stocks (binary).
- `returns`: Daily stock returns.
#### Output:
- Prints PnL correlation or raises an error if mismatched.

In [48]:
match_implementations(get_weights, benchmark_portfolio_vectorized, features, universe, returns)

Starting to generate Iterative Portfolio


100%|██████████| 41/41 [00:37<00:00,  1.10it/s]


Iterative Portfolio Generated
Correlation of 0.9991722687941328 between Iterative and Vectorized Implementations. Both implementations match!


### Final Notes
- We recommend using vectorized code to test out your strategies. This will be easier and faster to run but make sure to `shift(1)` feature dataframes in order to avoid lookahead or forward bias.
- At the end when you have decided your final strategy that you want to submit for the competition, we advise you to write code for `get_weights` which will help iteratively generate your portfolio. 
- Finally, before submitting make sure to run `match_implementations` to make sure that both versions of your code produce the same portfolio
- If these two portfolios match, you can submit the one which was produced by `generate_portfolio_vectorized` without waiting for the iterative portfolio. You don't need to run the `generate_portfolio` function for 20 years!
- We will check that the submission you made on Kaggle matches with the portfolio generated by your code. If these two don't match, you will be eliminated from the competition. 

In [49]:
# Submit this csv file on kaggle
full_portfolio.to_csv("submission.csv")